# Global spatially variable feature baselines in Python
**Moran's I / Geary's C (Squidpy) · SpatialDE · Hotspot**

These are run (a) on all cells, the global SVG baseline, and (b) **within each cell type**, the simplest ctSVP detector for single-cell imaging.
Comparing (a) with (b) shows the confounding that Li, Ge & Jiang (2025) describe: lineage markers dominate (a) and largely disappear from (b).

> Template, not executed. Install: `pip install squidpy spatialde naivede hotspotsc`

In [ ]:
import numpy as np, pandas as pd, scanpy as sc, squidpy as sq

adata = sq.datasets.imc()              # IMC breast cancer (Jackson et al. 2020 subset)
# or: adata = sc.read_h5ad("../../datasets/raw/squidpy_imc.h5ad")
print(adata)                           # obs['cell type'], obsm['spatial']
adata.X = np.arcsinh(adata.X / 5) if adata.X.max() > 50 else adata.X   # arcsinh transform if raw intensities

## 1. Moran's I (global)

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=10)
sq.gr.spatial_autocorr(adata, mode="moran", n_perms=100, n_jobs=4)
adata.uns["moranI"].head(10)           # I, pval_norm, pval_z_sim, pval_norm_fdr_bh

## 2. Moran's I **within each cell type** (simple ctSVP test)

In [ ]:
ct_key = "cell type"
rows = []
for ct, idx in adata.obs.groupby(ct_key).indices.items():
    if len(idx) < 50:
        continue                                        # too few cells
    sub = adata[idx].copy()
    sq.gr.spatial_neighbors(sub, coord_type="generic", n_neighs=10)
    sq.gr.spatial_autocorr(sub, mode="moran", n_perms=100, n_jobs=4)
    r = sub.uns["moranI"].assign(cell_type=ct)
    rows.append(r.reset_index(names="feature"))
ctsv = pd.concat(rows)
ctsv.sort_values("pval_norm").head(20)

**Caveat (imaging-specific):** lateral spillover from neighbouring cells of another type can create apparent within-type spatial structure. For example, CD3 signal leaking into tumour cells next to T cells.
One mitigation is to regress each marker on the neighbour composition before testing:

In [ ]:
# neighbour composition covariates (fraction of each cell type among the k nearest neighbours)
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
k = 10
nn = NearestNeighbors(n_neighbors=k + 1).fit(adata.obsm["spatial"])
_, ind = nn.kneighbors(adata.obsm["spatial"])
onehot = pd.get_dummies(adata.obs[ct_key]).to_numpy(float)
nbr_comp = onehot[ind[:, 1:]].mean(axis=1)              # cells x K
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
resid = X - LinearRegression().fit(nbr_comp, X).predict(nbr_comp)
adata.layers["resid_nbr"] = resid                       # rerun the within-type test on this layer

## 3. SpatialDE (Gaussian process)

In [ ]:
import SpatialDE
coords = pd.DataFrame(adata.obsm["spatial"], columns=["x", "y"], index=adata.obs_names)
expr   = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)
# SpatialDE is O(n^3): subsample or run per image / per cell type
sub = np.random.default_rng(0).choice(adata.n_obs, size=min(3000, adata.n_obs), replace=False)
sde = SpatialDE.run(coords.iloc[sub], expr.iloc[sub])
sde.sort_values("qval").head(10)[["g", "l", "FSV", "pval", "qval"]]

## 4. Hotspot (kNN autocorrelation; `normal` model for continuous protein data)

In [ ]:
import hotspot
adata.layers["dense"] = X
hs = hotspot.Hotspot(adata, layer_key="dense", model="normal", latent_obsm_key="spatial")
hs.create_knn_graph(weighted_graph=False, n_neighbors=30)
hs_res = hs.compute_autocorrelations(jobs=4)
hs_res.head(10)                                          # C (autocorrelation), Z, Pval, FDR
# modules of co-varying markers:
lc = hs.compute_local_correlations(hs_res.index[:20], jobs=4)
modules = hs.create_modules(min_gene_threshold=3, core_only=True, fdr_threshold=0.05)

## 5. Save in the common long format for `05_evaluation_framework`

In [ ]:
ctsv.rename(columns={"pval_norm": "p_value"}).assign(method="MoranI_within_type")[
    ["method", "cell_type", "feature", "p_value"]].to_csv("results_moran_within_type.csv", index=False)